In [4]:
import os
import pandas as pd

In [5]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
from lets_plot import * 
LetsPlot.setup_html()

In [7]:
pip install lets-plot


Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [10]:
import numpy as np
import pandas as pd

from lets_plot import *
LetsPlot.setup_html()

In [11]:
filename = 'waitrose/bakery.csv' # Can you change this to the file you want to read?
df = pd.read_csv('../data/bakery.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 485 entries, 0 to 484
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   data-product-id        485 non-null    int64 
 1   data-product-name      485 non-null    object
 2   data-product-type      485 non-null    object
 3   data-product-on-offer  485 non-null    bool  
 4   data-product-index     485 non-null    int64 
 5   image-url              485 non-null    object
 6   product-page           485 non-null    object
 7   product-name           485 non-null    object
 8   product-size           482 non-null    object
 9   item-price             485 non-null    object
 10  price-per-unit         463 non-null    object
 11  offer-description      52 non-null     object
 12  category               485 non-null    object
dtypes: bool(1), int64(2), object(10)
memory usage: 46.1+ KB


In [12]:
# Drop duplicates
df = df.drop_duplicates()

df = df.drop(columns=['data-product-name', 
                      'data-product-type', 
                      'data-product-index', 
                      'category'])
df = (
    df.rename(columns={
        'data-product-id': 'id',
        'data-product-price': 'price',
        'data-product-on-offer': 'offer',
        'product-page': 'page',
        'product-name': 'name',
        'product-size': 'size',
    })
)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 482 entries, 0 to 484
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 482 non-null    int64 
 1   offer              482 non-null    bool  
 2   image-url          482 non-null    object
 3   page               482 non-null    object
 4   name               482 non-null    object
 5   size               479 non-null    object
 6   item-price         482 non-null    object
 7   price-per-unit     460 non-null    object
 8   offer-description  49 non-null     object
dtypes: bool(1), int64(1), object(7)
memory usage: 34.4+ KB


In [13]:
# The id does not need 64 bits. 32 bits is enough.
# See https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.intc for ranges.
df['id'] = df['id'].astype('int32')

In [14]:
df.loc[df['item-price'].str.contains('p'), 'item-price'] = df['item-price'].apply(lambda x: '0.' + str.replace(x, 'p', ''))
df.loc[df['item-price'].str.contains('£'), 'item-price'] = df['item-price'].str.replace('£', '')
df['item-price'] = df['item-price'].astype('float')

In [15]:
df['item-price'].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
item-price,482.0,4.84668,7.750208,0.5,1.6,2.2,3.15,45.0


In [16]:
all_bread = df['name'].str.contains('bread', case=False)

print(f"There are {all_bread.sum()} bread products in the dataset.")

There are 52 bread products in the dataset.


In [17]:
# Follow my live demo to understand the process of writing the code below.
df[all_bread][['name', 'size', 'item-price', 'page']].sort_values(['name', 'size']).set_index(['page'])

,name,size,item-price
page,,,
https://www.waitrose.com/ecom/products/all-butter-shortbread/056597-28405-28406,All Butter Shortbread,each,1.20
https://www.waitrose.com/ecom/products/bfree-gluten-free-wholegrain-pitta-breads/657631-695118-695119,BFree Gluten Free Wholegrain Pitta Breads,4x55g,2.90
https://www.waitrose.com/ecom/products/bacheldr-rustic-crunch-bread-mix/559129-220498-220499,Bacheldr Rustic Crunch Bread Mix,500g,1.50
https://www.waitrose.com/ecom/products/cohens-bakery-buckingham-rye-bread/077133-39207-39208,Cohens Bakery Buckingham Rye Bread,400g,2.10
https://www.waitrose.com/ecom/products/crosta-mollica-piadina-flatbreads/817933-198092-198093,Crosta & Mollica Piadina Flatbreads,3s,2.00
https://www.waitrose.com/ecom/products/crosta-mollica-wholeblend-flatbreads/807541-307196-307197,Crosta & Mollica Wholeblend Flatbreads,4s,2.00
https://www.waitrose.com/ecom/products/deli-kitchen-greek-style-flatbreads/831341-702286-702287,Deli Kitchen Greek Style Flatbreads,4s,1.90
https://www.waitrose.com/ecom/products/deli-kitchen-plain-folded-flatbreads/521824-492967-492968,Deli Kitchen Plain Folded Flatbreads,6s,1.50
https://www.waitrose.com/ecom/products/deli-kitchen-seeded-folded-flatbreads/884168-493355-493356,Deli Kitchen Seeded Folded Flatbreads,6s,1.50


In [18]:

breads_to_remove = ['shortbread', 'flatbread', 'bread mix', 'pitta bread', 'gingerbread']

# Rebuild all_breads to exclude the breads_to_remove
all_bread = df['name'].str.contains('bread', case=False) & ~df['name'].str.contains('|'.join(breads_to_remove), case=False)

print(f"There are {all_bread.sum()} bread products in the dataset.")

There are 31 bread products in the dataset.


In [19]:
df_bread = df[all_bread].sort_values(['name', 'size'])

df_bread[['name', 'size', 'item-price', 'page']].set_index(['page'])

,name,size,item-price
page,,,
https://www.waitrose.com/ecom/products/cohens-bakery-buckingham-rye-bread/077133-39207-39208,Cohens Bakery Buckingham Rye Bread,400g,2.10
https://www.waitrose.com/ecom/products/essential-white-medium-sliced-bread/055018-27631-27632,Essential White Medium Sliced Bread,800g,0.75
https://www.waitrose.com/ecom/products/essential-wholemeal-medium-sliced-bread/055051-27670-27671,Essential Wholemeal Medium Sliced Bread,800g,0.75
https://www.waitrose.com/ecom/products/hovis-1886-granary-sliced-bread/841477-746623-746624,Hovis 1886 Granary Sliced Bread,450g,1.50
https://www.waitrose.com/ecom/products/hovis-1886-seeded-sliced-bread/531774-746615-746616,Hovis 1886 Seeded Sliced Bread,450g,1.50
https://www.waitrose.com/ecom/products/hovis-best-of-both-medium-sliced-bread/084740-42991-42992,Hovis Best of Both Medium Sliced Bread,800g,1.35
https://www.waitrose.com/ecom/products/hovis-granary-wholemeal-sliced-bread/054266-27239-27240,Hovis Granary Wholemeal Sliced Bread,800g,1.90
https://www.waitrose.com/ecom/products/hovis-original-granary-medium-sliced-bread/055416-27817-27818,Hovis Original Granary Medium Sliced Bread,800g,1.90
https://www.waitrose.com/ecom/products/hovis-original-granary-thick-sliced-bread/055312-27767-27768,Hovis Original Granary Thick Sliced Bread,400g,1.25


In [20]:
# This is the simpler solution, but why does it look odd and different to the data we've been seeing in previous steps?
df_bread.groupby('name')['size'].unique()

name
Cohens Bakery Buckingham Rye Bread                      [400g]
Essential White Medium Sliced Bread                     [800g]
Essential Wholemeal Medium Sliced Bread                 [800g]
Hovis 1886 Granary Sliced Bread                         [450g]
Hovis 1886 Seeded Sliced Bread                          [450g]
Hovis Best of Both Medium Sliced Bread                  [800g]
Hovis Granary Wholemeal Sliced Bread                    [800g]
Hovis Original Granary Medium Sliced Bread              [800g]
Hovis Original Granary Thick Sliced Bread               [400g]
Hovis Seed Sensations Multiseeded Sliced Bread    [400g, 800g]
Hovis Soft White Medium Sliced White Bread              [800g]
Hovis Soft White Thick Sliced White Bread               [800g]
Hovis Wholemeal Medium Sliced Bread               [400g, 800g]
Hovis Wholemeal Thick Sliced Bread                      [800g]
Irwin's Together Brown Soda Bread                       [400g]
Livlife Seriously Seeded Sliced Bread             

In [21]:
df_bread.groupby('name')['size'].nunique()

name
Cohens Bakery Buckingham Rye Bread                1
Essential White Medium Sliced Bread               1
Essential Wholemeal Medium Sliced Bread           1
Hovis 1886 Granary Sliced Bread                   1
Hovis 1886 Seeded Sliced Bread                    1
Hovis Best of Both Medium Sliced Bread            1
Hovis Granary Wholemeal Sliced Bread              1
Hovis Original Granary Medium Sliced Bread        1
Hovis Original Granary Thick Sliced Bread         1
Hovis Seed Sensations Multiseeded Sliced Bread    2
Hovis Soft White Medium Sliced White Bread        1
Hovis Soft White Thick Sliced White Bread         1
Hovis Wholemeal Medium Sliced Bread               2
Hovis Wholemeal Thick Sliced Bread                1
Irwin's Together Brown Soda Bread                 1
Livlife Seriously Seeded Sliced Bread             1
No.1 Malt Sourdough Bread with Seeds              1
No.1 Rye and Wheat Dark Sourdough Bread           1
No.1 Spelt Sourdough Bread                        1
No.1 Wh

In [22]:
dum = df_bread.groupby('name').apply(
    lambda x: pd.Series({
        'available_size': x['size'].unique(),
        'num_size': x['size'].nunique()

    })
).reset_index()
dum.sort_values(by= 'num_size',ascending = False).head(5)

C:\Users\Oviya S\AppData\Local\Temp\ipykernel_8772\2999231855.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dum = df_bread.groupby('name').apply(


,name,available_size,num_size
12,Hovis Wholemeal Medium Sliced Bread,"[400g, 800g]",2
9,Hovis Seed Sensations Multiseeded Sliced Bread,"[400g, 800g]",2
2,Essential Wholemeal Medium Sliced Bread,[800g],1
1,Essential White Medium Sliced Bread,[800g],1
0,Cohens Bakery Buckingham Rye Bread,[400g],1


In [23]:
df_bread['is_sliced'] = df_bread['name'].str.contains('sliced',case = False)
df_bread.head()

,id,offer,image-url,page,name,size,item-price,price-per-unit,offer-description,is_sliced
484,77133,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/cohens-...,Cohens Bakery Buckingham Rye Bread,400g,2.10,52.5p/100g,NaN,False
178,55018,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/essenti...,Essential White Medium Sliced Bread,800g,0.75,9.4p/100g,NaN,True
79,55051,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/essenti...,Essential Wholemeal Medium Sliced Bread,800g,0.75,9.4p/100g,NaN,True
227,841477,True,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/hovis-1...,Hovis 1886 Granary Sliced Bread,450g,1.50,33.3p/100g,save 30p. Was £1.80,True
61,531774,True,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/hovis-1...,Hovis 1886 Seeded Sliced Bread,450g,1.50,33.3p/100g,save 30p. Was £1.80,True


In [24]:
# Convert size to numeric grams
df_bread['size_g'] = df_bread['size'].str.replace('g','').astype(float)

# Create price per kg column
df_bread['price-per-kg'] = (df_bread['item-price']/ df_bread['size_g'])*1000

# Remove help column
df_bread.drop(columns=['size_g'],inplace= True)

# Check result
df_bread[['name','size','item-price','price-per-kg']].head()
df_bread.head()

,id,offer,image-url,page,name,size,item-price,price-per-unit,offer-description,is_sliced,price-per-kg
484,77133,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/cohens-...,Cohens Bakery Buckingham Rye Bread,400g,2.10,52.5p/100g,NaN,False,5.250000
178,55018,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/essenti...,Essential White Medium Sliced Bread,800g,0.75,9.4p/100g,NaN,True,0.937500
79,55051,False,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/essenti...,Essential Wholemeal Medium Sliced Bread,800g,0.75,9.4p/100g,NaN,True,0.937500
227,841477,True,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/hovis-1...,Hovis 1886 Granary Sliced Bread,450g,1.50,33.3p/100g,save 30p. Was £1.80,True,3.333333
61,531774,True,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/hovis-1...,Hovis 1886 Seeded Sliced Bread,450g,1.50,33.3p/100g,save 30p. Was £1.80,True,3.333333


In [25]:
print(df_bread[['name','prize,'size','price-per-kg','is_sliced']])

SyntaxError: unterminated string literal (detected at line 1) (2166050850.py, line 1)

In [26]:
print(df_bread.columns)

Index(['id', 'offer', 'image-url', 'page', 'name', 'size', 'item-price',
       'price-per-unit', 'offer-description', 'is_sliced', 'price-per-kg'],
      dtype='object')
